# 06. Interactive Networks with Plotly (and friends)

This is the **interactive counterpart** to notebooks 01–05. You have already built a complete static pipeline: `nx.draw_*` basics (01), overlap-free layouts (02), principled labelling (03), community-aware composites (04), and spatial networks (05). This notebook does not re-teach any of that — it asks the next question:

> Once the static figure is right, what does **interactivity** add, and what does it cost?

You'll see why Plotly builds networks from *separate* edge and node traces (no `nx.draw()` shortcut here), how hover replaces most labels, how `updatemenus` and `sliders` let readers switch layouts or filter communities on the fly, and when to reach for **Pyvis** or **Dash Cytoscape** instead. We close with a flagship interactive version of the Game of Thrones case study from notebook 04.

**Roadmap**

1. What interactivity buys — and what it doesn't (bridge from the static notebooks).
2. The mental model shift: traces, not draw calls.
3. First interactive network: rebuild a static figure in Plotly.
4. Hover, colour, size — surface metadata without extra ink.
5. Switching layouts interactively with `updatemenus`.
6. Edge attributes and directedness — weights, arrows, neighbourhood highlighting.
7. Controls and filtering — dropdowns, sliders, buttons.
8. Alternative library: Pyvis (and when Dash Cytoscape wins instead).
9. Flagship case study — the GOT notebook-04 figure, now interactive.
10. Scaling and pitfalls — performance, accessibility, when *not* to go interactive.
11. Further resources.

## Requirements

Beyond the course-wide stack (NumPy, pandas, NetworkX, Matplotlib), this notebook uses:

| Package | Version tested | Purpose |
|---|---|---|
| `plotly` | `≥ 5.20` | interactive figures and widgets |
| `pyvis` | `≥ 0.3.2` | physics-based HTML exports (Section 8) |
| `ipywidgets` | `≥ 8.0` | `FigureWidget` callbacks |

Install (uncomment if needed):

In [ ]:
# !pip install plotly pyvis ipywidgets

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots

cwd = Path.cwd().resolve()
tutorials_dir = cwd.parent if cwd.name == "04-networks" else cwd / "tutorials"
if str(tutorials_dir) not in sys.path:
    sys.path.insert(0, str(tutorials_dir))

from dataviz_utils import *

set_seeds()


def to_css(color):
    """Normalise a colour to a Plotly-accepted string.

    `lighten_color` from dataviz_utils returns an RGB tuple of floats in
    [0, 1], which Plotly rejects — it expects hex / rgb() / named colours.
    """
    if isinstance(color, str):
        return color
    r, g, b = [int(round(float(c) * 255)) for c in color[:3]]
    return f"rgb({r},{g},{b})"


NOTEBOOK_DIR = tutorials_dir / "04-networks"
DATA_DIR = NOTEBOOK_DIR / "data"
SEED = 42

## 1. What interactivity buys — and what it doesn't

**Learning objectives**
- name three limitations of the static notebooks that interactivity can relieve;
- name two cases where static is still the right answer.

### 🔁 From Static to Interactive

Recall the specific trade-offs from notebooks 02–04:

- **Notebook 02** spent a whole section measuring *node-marker overlap in rendered pixels*, because once a figure is exported to PDF, overlapping nodes are locked in.
- **Notebook 03** carefully picked **which 8 labels out of 187** to show, because drawing them all produced a spaghetti of text.
- **Notebook 04**'s flagship composite used a legend band, a histogram, and a ranking panel — three extra panels — just to let the reader triangulate colour → name → centrality.

Interactivity collapses those trade-offs:

| Static limitation | Interactive answer |
|---|---|
| only 8 labels fit | hover shows a label *only when the reader asks* |
| legend panel needed | tooltip carries the metadata — no separate key |
| one layout per figure | a dropdown switches layouts without re-running code |
| 187 nodes feel dense | zoom/pan lets the reader choose the detail level |

But interactivity is not free. It fails in three situations:

1. **Print / PDF handouts** — readers can't hover on paper.
2. **Teaching a specific finding** — a static annotation is unambiguous; a tooltip is optional.
3. **Large graphs (>5k nodes)** — Plotly SVG rendering slows down; you'd reach for WebGL (`scattergl`) or a dedicated library.

**Rule of thumb**: static for *claims*, interactive for *exploration*.

## 2. The mental model shift — traces, not `nx.draw()`

**Learning objectives**
- explain why Plotly does not ship an `nx.draw()` equivalent;
- build a minimal network from scratch as two `go.Scatter` traces.

### Why Plotly has no single `nx.draw()` call

Matplotlib's `nx.draw_networkx_*` lives in the *imperative* world: each call adds a Matplotlib artist to an axes. Plotly lives in the *declarative* world: a **figure is a JSON document** describing traces and layout. There is no "artist" to add, so NetworkX can't offer a one-liner.

The upside: since the figure is just data, anything you build can be filtered, restyled, or animated by handing Plotly a different JSON payload. That's what makes `updatemenus` and `sliders` possible in Section 5.

### The two-trace pattern

Every Plotly network figure reduces to:

- **one edge trace** — a single `go.Scatter` with `mode="lines"`. Edges are concatenated with `None` separators so all of them render in one call (fast).
- **one node trace** — a `go.Scatter` with `mode="markers"` (plus `+text` if labels are shown).

Nothing more. All community colour, hover metadata, size-by-centrality decisions live on the node trace as vectors. The edge trace stays dumb.

### ⚠️ Common mistake

> *"I'll add one Scatter per edge so I can style each one independently."*

For a 684-edge graph (GOT Book 1) that creates 684 SVG elements and slows the page dramatically. If you need edges of different widths or colours, **bin them** into a handful of traces (e.g. 4 weight quantiles, as we do in Section 6).

In [ ]:
# A 6-node toy graph to isolate the two-trace pattern.
toy = nx.cycle_graph(6)
toy.add_edge(0, 3)  # a chord so it's not a pure cycle
toy_pos = nx.spring_layout(toy, seed=SEED)

# --- Edge trace: one Scatter with None-separated segments ---
edge_x, edge_y = [], []
for u, v in toy.edges():
    edge_x += [toy_pos[u][0], toy_pos[v][0], None]
    edge_y += [toy_pos[u][1], toy_pos[v][1], None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    mode="lines",
    line=dict(width=1.2, color="#B0B8C1"),
    hoverinfo="none",  # edges are usually too short to hover on
)

# --- Node trace: one Scatter with per-point colour / size / hover ---
node_x = [toy_pos[n][0] for n in toy.nodes()]
node_y = [toy_pos[n][1] for n in toy.nodes()]

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode="markers+text",
    text=[str(n) for n in toy.nodes()],
    textposition="top center",
    marker=dict(size=22, color=DV_PALETTE["blue"], line=dict(color="black", width=1)),
    hovertemplate="node %{text}<extra></extra>",
)

fig = go.Figure([edge_trace, node_trace])
fig.update_layout(
    title="Minimal Plotly network — two traces, full stop",
    xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"),
    width=520, height=440, margin=dict(l=20, r=20, t=50, b=20),
    showlegend=False,
)
fig

**How to read the output.** Hover any node and Plotly reads `hovertemplate` off the node trace. Edges are invisible to hover (`hoverinfo="none"`) — they exist only as a background sketch.

### Key takeaways
- A Plotly network = **one edge trace + one node trace**, by default.
- `None`-separated coordinates let a single `Scatter` draw all edges cheaply.
- Style vectors (`marker.color`, `marker.size`, `customdata`) live on the node trace.

### Exercises
1. Rebuild the toy graph with `mode="markers"` only (no labels). Which is easier to read at this scale — markers + text or markers + hover?
2. Replace `go.Scatter` with `go.Scattergl` and check whether the figure still renders correctly. (WebGL backend — we'll return to this in Section 10.)
3. Change the `None` separator to `np.nan`. Does Plotly handle both? Why might `None` be safer for list-of-mixed-types pipelines?

## 3. First interactive network — the GOT graph, reloaded

**Learning objectives**
- load the same Book 1 edge list used in notebooks 02–04;
- wrap the two-trace pattern in a reusable helper;
- produce the first *real* interactive figure of the notebook.

### 🔁 From Static to Interactive

In notebook 02 we laid this graph out with `graphviz_layout(prog="neato", args="-Goverlap=prism")` specifically to avoid node-marker overlap in the rendered PDF. In Plotly the problem partly dissolves — the reader can zoom. But the layout still has to be *readable at default zoom*, so we keep using the same overlap-free layout.

In [ ]:
got_edges = pd.read_csv(DATA_DIR / "got" / "book1.csv")
GOT = nx.from_pandas_edgelist(
    got_edges, source="Source", target="Target",
    edge_attr="weight", create_using=nx.Graph(),
)

# Same layout as notebook 02 — overlap-free, weight-aware.
try:
    got_pos = nx.nx_agraph.graphviz_layout(GOT, prog="neato", args="-Goverlap=prism")
except (ImportError, OSError):
    # Graphviz not installed? Fall back to spring layout. Good enough for this
    # notebook, since the point is interactivity, not layout quality.
    got_pos = nx.spring_layout(GOT, weight="weight", seed=SEED)

# Normalise positions into [-1, 1] so every section starts from the same frame.
_coords = np.array(list(got_pos.values()))
_scale = np.ptp(_coords, axis=0).max() / 2
_centre = _coords.mean(axis=0)
got_pos = {n: ((x - _centre[0]) / _scale, (y - _centre[1]) / _scale)
           for n, (x, y) in got_pos.items()}

print(f"{GOT.number_of_nodes()} nodes, {GOT.number_of_edges()} edges")

In [ ]:
def edges_xy(G, pos, edgelist=None):
    """Return (x, y) lists with None-separators, ready for go.Scatter."""
    edgelist = G.edges() if edgelist is None else edgelist
    xs, ys = [], []
    for u, v in edgelist:
        xs += [pos[u][0], pos[v][0], None]
        ys += [pos[u][1], pos[v][1], None]
    return xs, ys


def make_edge_trace(G, pos, *, width=0.8, color="#C8CED8", name="edges"):
    xs, ys = edges_xy(G, pos)
    return go.Scatter(x=xs, y=ys, mode="lines",
                      line=dict(width=width, color=color),
                      hoverinfo="none", name=name, showlegend=False)


def make_node_trace(G, pos, *, size=10, color=DV_PALETTE["blue"], text=None,
                   hovertemplate=None, customdata=None, name="nodes"):
    nodes = list(G.nodes())
    return go.Scatter(
        x=[pos[n][0] for n in nodes],
        y=[pos[n][1] for n in nodes],
        mode="markers",
        marker=dict(size=size, color=color,
                    line=dict(color="black", width=0.6)),
        text=text if text is not None else nodes,
        customdata=customdata,
        hovertemplate=hovertemplate,
        name=name, showlegend=False,
    )


def blank_axes(fig, **kwargs):
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)
    fig.update_layout(plot_bgcolor="white", **kwargs)
    return fig

In [ ]:
fig = go.Figure([
    make_edge_trace(GOT, got_pos),
    make_node_trace(GOT, got_pos, size=8, hovertemplate="%{text}<extra></extra>"),
])
blank_axes(fig, title="GOT Book 1 — same graph, same layout, now interactive",
           width=760, height=640, margin=dict(l=20, r=20, t=50, b=20))
fig

**How to read the output.** You can pan (click-drag), zoom (scroll or box-select), and hover any node to see its name. No labels are drawn — hover replaced them.

### Key takeaways
- The same `got_pos` we used in notebook 02 works unchanged; only the *rendering* changed.
- Three tiny helpers (`edges_xy`, `make_edge_trace`, `make_node_trace`) will carry us through every figure in this notebook.
- Hover eliminates the "which 8 labels?" selection problem from notebook 03 by making *every* label conditionally visible.

### Exercises
1. Swap the Graphviz layout for `nx.spring_layout(..., seed=42)` and compare readability at the default zoom level.
2. Add `text=list(GOT.nodes())` + `mode="markers+text"` to the node trace. At what zoom does the label clutter from notebook 03 reappear?
3. Measure interactive load time: `%time fig.show()` vs. a Matplotlib `plt.savefig` of the same graph. Where does the time go?

## 4. Hover, colour, size — surfacing metadata interactively

**Learning objectives**
- encode community via colour and centrality via size (like notebook 04);
- attach structured metadata (degree, community, betweenness) to each node;
- write a `hovertemplate` that reads the metadata as a tooltip card.

### 🔁 From Static to Interactive

In notebook 04 the flagship figure needed a *legend panel*, a *histogram panel*, and a *ranking panel* so readers could triangulate colour → community name → centrality. Interactively, all three can live in the **tooltip of each node**.

In [ ]:
# Reproduce the three channels from notebook 04.
communities = nx.community.louvain_communities(GOT, seed=SEED, weight="weight")
communities = sorted(communities, key=len, reverse=True)
node_to_cid = {n: cid for cid, comm in enumerate(communities) for n in comm}

COMMUNITY_NAMES_BY_ANCHOR = {
    "Eddard-Stark":       "King's Landing Court",
    "Tyrion-Lannister":   "Riverlands & the Eyrie",
    "Jon-Snow":           "The Night's Watch",
    "Sansa-Stark":        "Sansa in King's Landing",
    "Daenerys-Targaryen": "Daenerys & the Dothraki",
    "Bran-Stark":         "Winterfell",
    "Waymar-Royce":       "Prologue — Beyond the Wall",
    "Danwell-Frey":       "House Frey",
}

wd = dict(GOT.degree(weight="weight"))
btw = nx.betweenness_centrality(GOT, weight="weight", seed=SEED)

community_anchors = [
    max(c, key=lambda n: (wd[n], GOT.degree(n), n)) for c in communities
]
community_names = [COMMUNITY_NAMES_BY_ANCHOR.get(a, f"Arc around {a}")
                   for a in community_anchors]

### 💡 Tip — structured `customdata`

`customdata` is a per-point payload (any 2-D array). Inside `hovertemplate`, `%{customdata[0]}` indexes columns. This is the clean way to push "tooltip-only metadata" into the trace without polluting `text` (which can also affect label rendering).

In [ ]:
nodes = list(GOT.nodes())
palette = CATEGORICAL_PALETTE
node_color = [palette[node_to_cid[n] % len(palette)] for n in nodes]

# Marker size ∝ √betweenness  — same idea as notebook 04.
_btw = np.array([btw[n] for n in nodes])
node_size = 6 + 28 * np.sqrt(_btw / max(_btw.max(), 1e-9))

# customdata columns: degree, weighted degree, community name, betweenness
custom = np.stack([
    [GOT.degree(n) for n in nodes],
    [wd[n] for n in nodes],
    [community_names[node_to_cid[n]] for n in nodes],
    _btw,
], axis=-1)

hover = (
    "<b>%{text}</b><br>"
    "degree: %{customdata[0]}<br>"
    "weighted degree: %{customdata[1]}<br>"
    "community: %{customdata[2]}<br>"
    "betweenness: %{customdata[3]:.3f}"
    "<extra></extra>"
)

fig = go.Figure([
    make_edge_trace(GOT, got_pos),
    make_node_trace(
        GOT, got_pos,
        size=node_size, color=node_color,
        customdata=custom, hovertemplate=hover,
    ),
])
blank_axes(fig,
           title="GOT — colour = community, size = betweenness, hover = everything else",
           width=820, height=680, margin=dict(l=20, r=20, t=50, b=20))
fig

**How to read the output.** The biggest dots are the book's *brokers* (highest betweenness — Eddard, Tyrion, Jon). Hover one and you get its community name directly; no need to cross-reference a colour key.

### Key takeaways
- `customdata` carries per-node metadata without touching the visual channel.
- `%{text}` pulls from the `text=` argument; `%{customdata[i]}` pulls from the array.
- `<extra></extra>` suppresses Plotly's default trace-name annotation in the tooltip.

### Exercises
1. Add a 5th column to `customdata` holding each node's *rank* by betweenness (1 = highest). Display it as `rank #X of 187` in the tooltip.
2. Swap size encoding from `sqrt(betweenness)` to linear `betweenness`. Do the brokers still stand out or does the distribution's skew swallow them?
3. Colour by `wd` (weighted degree, continuous) using `colorscale="Viridis"` instead of community. When is continuous colour preferable to categorical?

## 5. Switching layouts interactively — `updatemenus`

**Learning objectives**
- pre-compute several layouts and attach them to one figure;
- use `updatemenus` to restyle `x`/`y` vectors on click;
- explain why interactivity is *pedagogically* useful here — layout is a modelling choice, not a fact.

### 🔁 From Static to Interactive

Notebook 02 argued that there is no "best" layout — only the right layout *for the question being asked*. Statically, we had to pick one. Interactively, the reader can toggle between them and **see** how interpretation shifts.

In [ ]:
def normalise(pos):
    coords = np.array(list(pos.values()))
    scale = np.ptp(coords, axis=0).max() / 2
    centre = coords.mean(axis=0)
    return {n: ((x - centre[0]) / scale, (y - centre[1]) / scale)
            for n, (x, y) in pos.items()}

layouts = {
    "Graphviz / overlap-free": got_pos,
    "spring (weight-aware)":   normalise(nx.spring_layout(GOT, weight="weight", seed=SEED)),
    "Kamada-Kawai":            normalise(nx.kamada_kawai_layout(GOT)),
    "circular":                normalise(nx.circular_layout(GOT)),
}

In [ ]:
# Build the figure with the *first* layout active.
name0, pos0 = next(iter(layouts.items()))
fig = go.Figure([
    make_edge_trace(GOT, pos0),
    make_node_trace(
        GOT, pos0,
        size=node_size, color=node_color,
        customdata=custom, hovertemplate=hover,
    ),
])

# Each updatemenu button restyles BOTH traces (indices 0 and 1) at once.
buttons = []
for name, pos in layouts.items():
    ex, ey = edges_xy(GOT, pos)
    nx_ = [pos[n][0] for n in GOT.nodes()]
    ny_ = [pos[n][1] for n in GOT.nodes()]
    buttons.append(dict(
        label=name, method="restyle",
        args=[{"x": [ex, nx_], "y": [ey, ny_]}, [0, 1]],
    ))

blank_axes(fig,
           title="One figure, four layouts — pick a lens",
           width=820, height=680, margin=dict(l=20, r=20, t=90, b=20),
           updatemenus=[dict(
               type="buttons", direction="right",
               x=0.0, y=1.09, xanchor="left", yanchor="top",
               buttons=buttons, showactive=True,
               pad=dict(l=2, r=2, t=2, b=2),
           )])
fig

**How to read the output.** Click each button: the nodes animate to the new layout. Notice how the *circular* layout destroys the community signal (neighbours scatter around the perimeter), while *spring* and *Kamada-Kawai* roughly agree with Graphviz on the group structure but disagree on the outliers.

### ⚠️ Common mistake — silent restyle failure

Plotly restyles **by trace index**. The `args=[{...}, [0, 1]]` form targets traces 0 and 1 *explicitly*. If you drop the index list, Plotly applies the same `x` and `y` to *all* traces, which silently breaks multi-trace figures (including the edge-bin figure in Section 6).

### Key takeaways
- `updatemenus` with `method="restyle"` swaps data in-place without rebuilding the figure.
- Always pass the trace-index list when more than one trace exists.
- Layout becomes an **exploratory parameter** instead of an editorial choice.

### Exercises
1. Add a fifth layout: a **community-aware ring layout** (reuse the helpers from notebook 04). Does it become the obvious favourite?
2. Swap `updatemenus` for a `sliders` control that animates from one layout to the next. Which metaphor fits layout-switching better — discrete buttons or a continuous slider?
3. What happens if two layouts have different node orders (e.g. you forget to iterate `GOT.nodes()` consistently)? Test it.

## 6. Edge attributes and directedness

**Learning objectives**
- bin edges by weight into multiple traces (a performance pattern);
- draw arrows on a directed subgraph;
- highlight a node's neighbourhood via a dropdown.

### 6.1 Binning edges by weight

A single edge trace can't carry per-edge widths or colours because `line.width` is scalar. The standard workaround: **bin edges into a handful of traces** (e.g. 4 weight quantiles).

In [ ]:
weights = np.array([d["weight"] for *_, d in GOT.edges(data=True)])
quantiles = np.quantile(weights, [0.25, 0.5, 0.75])

def bin_for(w):
    return int(np.digitize(w, quantiles))  # 0..3

bins = {0: [], 1: [], 2: [], 3: []}
for u, v, d in GOT.edges(data=True):
    bins[bin_for(d["weight"])].append((u, v))

bin_widths = [0.4, 0.9, 1.6, 2.6]
bin_alphas = ["#DBE0E6", "#B8C2CC", "#8A97A6", "#5B6B7A"]

edge_traces = []
for b in range(4):
    ex, ey = edges_xy(GOT, got_pos, edgelist=bins[b])
    edge_traces.append(go.Scatter(
        x=ex, y=ey, mode="lines",
        line=dict(width=bin_widths[b], color=bin_alphas[b]),
        hoverinfo="none",
        name=f"weight bin {b+1}", showlegend=False,
    ))

fig = go.Figure([
    *edge_traces,
    make_node_trace(GOT, got_pos, size=node_size, color=node_color,
                    customdata=custom, hovertemplate=hover),
])
blank_axes(fig, title="Edges binned by weight — heavier ties read darker/thicker",
           width=820, height=680, margin=dict(l=20, r=20, t=50, b=20))
fig

### 6.2 Arrows on a directed subgraph

Plotly's `go.Scatter` has no arrow-head primitive. The idiomatic trick is to use **figure `annotations`** — one per directed edge, anchored with `ax`/`ay` at the *source* and `x`/`y` at the *target*. This scales to a few hundred edges before it slows; for larger directed graphs, see Pyvis in §8.

The Game of Thrones co-appearance graph is *undirected*, so drawing arrows at random directions would be meaningless (and indeed, drawing arrows on an undirected graph is a classic beginner mistake). To make this demonstration **interpretable**, we orient each edge of Eddard Stark's ego network **toward the more-central endpoint**, using the node's degree in the *full* graph as a proxy for importance.

The recipe is useful beyond this example. Whenever you do have genuinely directed data — citation networks, migration flows, food webs, reply-to graphs, import graphs between code modules — the same `ax/ay → x/y` annotation pattern is how you render it in Plotly.

Two details worth copying out of the code below:

- **Make arrows visible.** A tiny `arrowwidth` and a pale `arrowcolor` will disappear against a plot background. Here we use `arrowhead=3` (filled triangular head), `arrowsize=2.0`, `arrowwidth=1.8`, and a dark `arrowcolor`.
- **Set `standoff` to the node radius.** The source/target endpoints sit at the centre of each node marker. Set `standoff` (pull back from the target) and `startstandoff` (pull back from the source) to roughly the marker radius in pixels so arrowheads don't disappear under the nodes.

In [ ]:
ego_center = "Eddard-Stark"
ego_nodes = set([ego_center]) | set(GOT.neighbors(ego_center))
ego = GOT.subgraph(ego_nodes).copy()
ego_pos = {n: got_pos[n] for n in ego.nodes()}

# Direct each ego edge toward the more-central endpoint.
# Use full-graph degree (not ego-subgraph degree) so the ranking reflects the
# character's standing in the whole story, not just in this small slice.
degrees = dict(GOT.degree())

directed_edges = []
for u, v in ego.edges():
    if degrees[v] >= degrees[u]:
        directed_edges.append((u, v))   # u --> v, toward the more central node
    else:
        directed_edges.append((v, u))

# Base layer: a muted undirected line for every edge, so the geometry of the
# subgraph is always readable even if a reader ignores the arrows.
ex, ey = edges_xy(ego, ego_pos)
edge_trace = go.Scatter(
    x=ex, y=ey, mode="lines",
    line=dict(width=1.0, color="#D5DDE7"),
    hoverinfo="none", showlegend=False,
)

# Arrow annotations — large enough to actually see, and with a standoff that
# keeps arrowheads clear of the node markers (which have radius ~ 11 px here).
NODE_SIZE = 22
edge_arrows = [
    dict(
        ax=ego_pos[u][0], ay=ego_pos[u][1],      # source
        x=ego_pos[v][0],  y=ego_pos[v][1],       # target
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=True,
        arrowhead=3,       # filled triangular head
        arrowsize=2.0,     # head-size multiplier
        arrowwidth=1.8,    # shaft width in px
        arrowcolor="#334155",
        standoff=12,       # pull back from target (node)
        startstandoff=12,  # pull back from source
        opacity=0.9,
    )
    for u, v in directed_edges
]

fig = go.Figure([
    edge_trace,
    make_node_trace(
        ego, ego_pos,
        size=NODE_SIZE,
        color=[DV_PALETTE["gold"] if n == ego_center else DV_PALETTE["blue"]
               for n in ego.nodes()],
        text=list(ego.nodes()),
        hovertemplate="%{text}<extra></extra>",
    ),
])
blank_axes(
    fig,
    title=(f"{ego_center} — ego network, "
           f"arrows point toward the more-central endpoint (full-graph degree)"),
    width=860, height=660, margin=dict(l=20, r=20, t=60, b=20),
    annotations=edge_arrows,
)
fig

### 6.3 Neighbourhood highlighting via a dropdown

Highlighting the neighbourhood of a picked node is the single most useful interactive affordance for dense networks — and it's *pure `updatemenus`*. For each choice of focal node we pre-compute which edges and which nodes should light up, and the dropdown swaps between those pre-computed states.

In [ ]:
focal_candidates = sorted(wd, key=lambda n: -wd[n])[:8]  # top 8 brokers

base_edge_trace = make_edge_trace(GOT, got_pos)  # dim background for context
focal_node_trace = make_node_trace(
    GOT, got_pos, size=node_size, color=node_color,
    customdata=custom, hovertemplate=hover,
)

# A second edge trace and a second node trace that we REPLACE on choice.
hilite_edge_trace = go.Scatter(
    x=[], y=[], mode="lines",
    line=dict(width=2.0, color=DV_PALETTE["red"]),
    hoverinfo="none", showlegend=False,
)
hilite_node_trace = go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(size=16, color=DV_PALETTE["red"],
                line=dict(color="black", width=1)),
    hoverinfo="skip", showlegend=False,
)

fig = go.Figure([base_edge_trace, focal_node_trace,
                 hilite_edge_trace, hilite_node_trace])

buttons = [dict(label="— none —", method="restyle",
                args=[{"x": [[], []], "y": [[], []]}, [2, 3]])]
for focal in focal_candidates:
    nbrs = list(GOT.neighbors(focal))
    ex, ey = edges_xy(GOT, got_pos, edgelist=[(focal, n) for n in nbrs])
    nx_ = [got_pos[focal][0]] + [got_pos[n][0] for n in nbrs]
    ny_ = [got_pos[focal][1]] + [got_pos[n][1] for n in nbrs]
    buttons.append(dict(
        label=focal, method="restyle",
        args=[{"x": [ex, nx_], "y": [ey, ny_]}, [2, 3]],
    ))

blank_axes(fig, title="Neighbourhood highlight — dropdown swaps the red overlay",
           width=820, height=700, margin=dict(l=20, r=20, t=90, b=20),
           updatemenus=[dict(
               type="dropdown", direction="down",
               x=0.0, y=1.09, xanchor="left", yanchor="top",
               buttons=buttons, showactive=True,
           )])
fig

**How to read the output.** Pick a broker and its immediate neighbours light up in red. This is the interactive answer to notebook 03's dilemma: instead of picking 8 permanent labels, the *reader picks the focal node* at runtime.

### Key takeaways
- **Bin** edges by weight when you need different widths or colours.
- Arrows live in `fig.layout.annotations`, not in traces.
- **Highlighting** = dropdown that swaps pre-computed overlay traces — no callback server needed.

### Exercises
1. Add a second dropdown that controls the *highlight colour*, so the reader can compare two focal nodes simultaneously.
2. For the arrows figure, switch to `arrowhead=3` and `arrowsize=2`. At what zoom do the arrowheads start to overlap the markers?
3. Make the edge-bins figure respect community — tint each bin by the *source community* colour instead of grey. What does it reveal?

## 7. Controls and filtering — sliders, dropdowns, buttons

**Learning objectives**
- filter edges by weight with a slider;
- toggle community visibility with a button row;
- know when to reach for `FigureWidget` (Python callbacks) vs `updatemenus` (pure JSON).

### 💡 Tip — `updatemenus` vs `FigureWidget`

- **`updatemenus` / `sliders`** run entirely in the browser and ship in exported HTML. Use them for *finite, pre-computable* states.
- **`FigureWidget`** runs Python callbacks — unbounded flexibility, but requires a live Jupyter kernel. Does not work in a static HTML export.

Prefer `updatemenus` whenever the set of states is finite.

In [ ]:
# --- Slider over a minimum-edge-weight threshold.
thresholds = [1, 2, 3, 5, 8, 12, 20]

fig = go.Figure([
    make_edge_trace(GOT, got_pos),
    make_node_trace(GOT, got_pos, size=node_size, color=node_color,
                    customdata=custom, hovertemplate=hover),
])

steps = []
for t in thresholds:
    kept = [(u, v) for u, v, d in GOT.edges(data=True) if d["weight"] >= t]
    ex, ey = edges_xy(GOT, got_pos, edgelist=kept)
    steps.append(dict(
        method="restyle",
        args=[{"x": [ex], "y": [ey]}, [0]],
        label=f"≥ {t}",
    ))

blank_axes(fig, title="Hide weak ties with a slider — does the backbone survive?",
           width=820, height=700, margin=dict(l=20, r=20, t=50, b=80),
           sliders=[dict(active=0, steps=steps, x=0.05, y=-0.02,
                        currentvalue=dict(prefix="min edge weight: "),
                        pad=dict(t=30))])
fig

In [ ]:
# --- Button row: solo / mute each community. One node trace per community so
# visibility can be toggled independently; one intra-community edge trace per
# community tinted by community colour; a single dim inter-community edge
# trace for the skeleton that stays visible in every state.

intra_traces, node_traces = [], []
for cid, members in enumerate(communities):
    subg = GOT.subgraph(members)
    ex, ey = edges_xy(subg, got_pos)
    intra_traces.append(go.Scatter(
        x=ex, y=ey, mode="lines",
        line=dict(width=1.0,
                  color=to_css(lighten_color(palette[cid % len(palette)], 0.5))),
        hoverinfo="none", showlegend=False, name=f"edges::{cid}",
    ))
    members_list = list(members)
    node_traces.append(go.Scatter(
        x=[got_pos[n][0] for n in members_list],
        y=[got_pos[n][1] for n in members_list],
        mode="markers",
        marker=dict(size=[node_size[nodes.index(n)] for n in members_list],
                    color=palette[cid % len(palette)],
                    line=dict(color="black", width=0.6)),
        text=members_list,
        hovertemplate="<b>%{text}</b><br>" +
                      f"community: {community_names[cid]}<extra></extra>",
        name=community_names[cid], showlegend=True,
    ))

inter_edges = [(u, v) for u, v, d in GOT.edges(data=True)
               if node_to_cid[u] != node_to_cid[v]]
ex, ey = edges_xy(GOT, got_pos, edgelist=inter_edges)
inter_trace = go.Scatter(x=ex, y=ey, mode="lines",
                         line=dict(width=0.6, color="#D8DEE5"),
                         hoverinfo="none", showlegend=False,
                         name="inter-community")

fig = go.Figure([inter_trace, *intra_traces, *node_traces])

n_comm = len(communities)
all_visible = [True] * (1 + 2 * n_comm)
buttons = [dict(label="all", method="update", args=[{"visible": all_visible}])]
for cid in range(n_comm):
    vis = [True]                                   # inter-community skeleton
    vis += [i == cid for i in range(n_comm)]       # intra-edges
    vis += [i == cid for i in range(n_comm)]       # nodes
    buttons.append(dict(label=f"solo: {community_names[cid]}",
                        method="update", args=[{"visible": vis}]))

blank_axes(fig, title="Solo a community — the legend is itself a control",
           width=900, height=700, margin=dict(l=20, r=220, t=80, b=20),
           showlegend=True,
           legend=dict(x=1.02, y=1.0, xanchor="left", yanchor="top"),
           updatemenus=[dict(
               type="dropdown", direction="down",
               x=0.0, y=1.08, xanchor="left", yanchor="top",
               buttons=buttons, showactive=True,
           )])
fig

**How to read the output.** Plotly legends are **clickable by default** — single-click hides a trace, double-click isolates it. That alone gives you free per-community visibility without an explicit dropdown. The dropdown above is still useful when you want to *guide* the reader toward meaningful comparisons.

### Key takeaways
- Sliders edit a single trace's `x`/`y`; buttons can edit any layout attribute (use `method="update"` for layout-level changes).
- One trace per community unlocks per-community `visible` toggles.
- Plotly's **legend is itself an interactive control** — remember to set `showlegend=True`.

### Exercises
1. Add a second slider that also filters by *minimum weighted degree* (hide nodes with `wd[n] < threshold`).
2. Combine the community solo button with the neighbourhood-highlight dropdown from Section 6. Does the UI stay readable?
3. Implement community-solo using Plotly's built-in legend double-click instead of a dropdown. Which UX is cleaner for an audience of peers vs. an audience of domain experts?

## 8. Alternative library — Pyvis

**Learning objectives**
- export a physics-driven interactive HTML with a dozen lines of Pyvis;
- compare Plotly and Pyvis on the same graph;
- know when to reach for Dash Cytoscape instead.

### Why Pyvis?

Pyvis wraps the **vis.js** JavaScript library. In return for giving up Plotly's `updatemenus` / `sliders` vocabulary, you get:

- a live **force-directed simulation** running in the browser;
- **drag-to-rearrange** nodes manually;
- **stable HTML export** — no Python kernel needed afterwards.

Pyvis is the right call for *teaching physics*, *demos*, and *distributable artefacts*. Plotly is the right call for *dashboards* and *publication-quality static exports*.

### 📚 Deeper dive — when Dash Cytoscape wins

Reach for **Dash Cytoscape** when you need:

- graph-specific selectors (e.g. CSS-style styling: `"node[class='broker']"`);
- tight integration with Python callbacks in a Dash app;
- compound nodes (nodes that visually contain other nodes);
- graph-theoretic layouts (`cose-bilkent`, `dagre`, `klay`) that Plotly doesn't ship.

The cost: you're now building a web application, not a notebook figure.

In [ ]:
try:
    from pyvis.network import Network
except ImportError:
    print("Pyvis not installed — skip this cell (see the requirements cell).")
else:
    net = Network(height="620px", width="100%", notebook=True,
                  cdn_resources="remote", bgcolor="#ffffff",
                  font_color="#222222")
    net.force_atlas_2based(gravity=-30, spring_length=90)

    for n in GOT.nodes():
        cid = node_to_cid[n]
        net.add_node(n, label=n, size=10 + 90 * np.sqrt(btw[n]),
                     color=palette[cid % len(palette)],
                     title=f"{n}<br>community: {community_names[cid]}"
                           f"<br>degree: {GOT.degree(n)}"
                           f"<br>weighted degree: {wd[n]}")
    for u, v, d in GOT.edges(data=True):
        net.add_edge(u, v, value=float(d["weight"]))

    out_path = NOTEBOOK_DIR / "cache" / "got_pyvis.html"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    net.write_html(str(out_path), notebook=False)
    print(f"Wrote {out_path.relative_to(NOTEBOOK_DIR.parent)}")

    from IPython.display import IFrame
    display(IFrame(src=str(out_path.relative_to(NOTEBOOK_DIR)),
                   width=820, height=640))

### Trade-off scorecard

| Need | Static (NetworkX + Matplotlib) | Plotly | Pyvis | Dash Cytoscape |
|---|---|---|---|---|
| PDF / print export | ✅ first-class | ⚠️ via Kaleido | ❌ HTML only | ❌ HTML only |
| hover tooltips | ❌ | ✅ `hovertemplate` | ✅ `title=` | ✅ tippy-style |
| drag / physics | ❌ | ❌ | ✅ native | ⚠️ via plugin |
| `updatemenus`/sliders | ❌ | ✅ | ❌ | ✅ via Dash |
| >5k nodes | 🚫 layout slow | ⚠️ `scattergl` helps | ⚠️ physics slow | ✅ WebGL |
| scripted look (CSS-like) | ✅ Matplotlib rc | ⚠️ JSON-y | ❌ | ✅ first-class |
| learning curve | gentle | moderate | easy | steep |

### Exercises
1. Take the Pyvis export and disable the physics simulation (`net.toggle_physics(False)`). Does the layout still make sense after you drag a few nodes?
2. Replicate the Plotly layout-switcher (Section 5) in Pyvis. You'll find it cannot — why?
3. Benchmark both: render the GOT graph in Plotly and Pyvis, time the *first paint* and the *interaction latency*. Where does each win?

## 9. Flagship case study — notebook 04's figure, now interactive

**Learning objectives**
- rebuild the notebook-04 multi-panel composite as a single interactive figure;
- pair the network view with supporting panels that update together;
- observe how interactivity collapses the static composite.

### 🔁 From Static to Interactive

Notebook 04's flagship was *four panels*: the network, a histogram, a ranking, and a legend band. Interactively, the histogram and ranking are really *hover overlays in disguise* — the reader can ask "where does this character sit in the distribution?" directly on the main panel, via the tooltip. We keep the histogram and ranking as **coordinated side panels** using `make_subplots`, because they answer *population-level* questions the hover can't: "how skewed is the interaction distribution?" and "who are the top 10 overall?"

In [ ]:
interaction_strength = pd.Series(wd).sort_values(ascending=False)
top10 = interaction_strength.head(10).sort_values()

fig = make_subplots(
    rows=2, cols=2,
    column_widths=[0.68, 0.32], row_heights=[0.58, 0.42],
    specs=[[{"type": "scatter", "rowspan": 2}, {"type": "histogram"}],
           [None,                              {"type": "bar"}]],
    horizontal_spacing=0.06, vertical_spacing=0.10,
    subplot_titles=("A. Characters and their communities",
                    "B. Interaction strength",
                    "C. Top-10 by weighted degree"),
)

# Panel A — network
fig.add_trace(make_edge_trace(GOT, got_pos), row=1, col=1)
fig.add_trace(make_node_trace(
    GOT, got_pos, size=node_size, color=node_color,
    customdata=custom, hovertemplate=hover,
), row=1, col=1)

# Panel B — histogram of interaction strength
fig.add_trace(go.Histogram(
    x=interaction_strength.values, nbinsx=24,
    marker_color=to_css(lighten_color(DV_PALETTE["blue"], 0.45)),
    marker_line=dict(color="#1F2933", width=0.6),
    hovertemplate="%{y} characters<br>%{x}<extra></extra>",
), row=1, col=2)
med = float(interaction_strength.median())
fig.add_vline(x=med, line=dict(color=DV_PALETTE["red"], width=1.4, dash="dash"),
              row=1, col=2)

# Panel C — top-10 horizontal bar
fig.add_trace(go.Bar(
    x=top10.values, y=top10.index, orientation="h",
    marker_color=DV_PALETTE["blue"],
    hovertemplate="%{y}: %{x}<extra></extra>",
), row=2, col=2)

# Network panel: kill ticks, square aspect.
fig.update_xaxes(visible=False, row=1, col=1)
fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_layout(
    title=dict(text="<b>A Song of Ice and Data</b> — Book 1, interactive edition",
               x=0.02, xanchor="left"),
    width=1080, height=720, plot_bgcolor="white", showlegend=False,
    margin=dict(l=20, r=20, t=80, b=40),
)
fig

**How to read the output.** Panel A is the same ring the notebook-04 figure lived on; hovering any node surfaces the same metadata that notebook 04 had to print in the legend band. Panels B and C now answer the *population-level* questions the reader might ask — *without extra text*, because the interactive titles and axes carry the rest.

### Key takeaways
- `make_subplots` composes a dashboard the same way notebook 04's `GridSpec` composed a static figure.
- Some static panels (legend band) **disappear** into hover; other panels (histogram, ranking) **remain** because they answer distributional questions hover can't.
- One shared `got_pos` + one shared colour scheme across panels keeps the reader oriented.

### Exercises
1. Add brushing: when the reader hovers a node in Panel A, highlight the same character's bar in Panel C. (Hint: `FigureWidget` + `on_hover`.)
2. Replace Panel B with a cumulative distribution (ECDF). Which view answers "the network is scale-free" better for a first-year student?
3. Export the whole composite to a standalone HTML (`fig.write_html(...)`) and share it. Does every tooltip still work without the Python kernel?

## 10. Scaling and pitfalls

**Learning objectives**
- know the size thresholds at which Plotly SVG slows down;
- pick between SVG (`go.Scatter`) and WebGL (`go.Scattergl`);
- check the accessibility floor of an interactive figure.

### Size thresholds (order-of-magnitude, not rules)

| Nodes / edges | SVG (`Scatter`) | WebGL (`Scattergl`) |
|---|---|---|
| ≤ 500 / ≤ 2k | instant | overkill |
| ~2k / ~10k | noticeable pan lag | smooth |
| ~10k / ~50k | unusable | smooth, but hover selection slows |
| > 10k | 🚫 don't use Plotly | consider `datashader` + static image |

### ⚠️ Common mistakes at scale

1. **One Scatter per edge.** Already discussed — never do this above ~200 edges.
2. **Per-edge hover.** Edges are thin; hovering an edge is fiddly even for the reader. Set `hoverinfo="none"` on edge traces by default.
3. **Forgetting `uirevision`.** When you update data behind the scenes, the user's pan/zoom resets. Setting `uirevision="stable"` on the layout preserves the camera across updates.
4. **Ignoring accessibility.** Plotly is SVG-based, so tooltips are screen-readable *in theory*, but colour-only encoding still fails for colourblind readers. Use `CATEGORICAL_PALETTE` (Okabe-Ito) and always back up colour with text in the tooltip.

In [ ]:
# Quick demonstration: a 2000-node scale-free graph via WebGL.
big = nx.barabasi_albert_graph(2000, 2, seed=SEED)
big_pos = nx.spring_layout(big, seed=SEED)

bx, by = edges_xy(big, big_pos)
big_nodes = list(big.nodes())

fig = go.Figure([
    go.Scattergl(x=bx, y=by, mode="lines",
                 line=dict(width=0.4, color="#D0D6DE"),
                 hoverinfo="none"),
    go.Scattergl(
        x=[big_pos[n][0] for n in big_nodes],
        y=[big_pos[n][1] for n in big_nodes],
        mode="markers",
        marker=dict(size=4, color=DV_PALETTE["blue"], line=dict(width=0)),
        hovertemplate="deg %{customdata}<extra></extra>",
        customdata=[big.degree(n) for n in big_nodes],
    ),
])
blank_axes(fig, title="2000-node Barabási-Albert via WebGL (go.Scattergl)",
           width=780, height=560, margin=dict(l=20, r=20, t=50, b=20),
           uirevision="stable")
fig

### Key takeaways
- Below ~1k nodes, prefer SVG (`Scatter`) — crisper rendering, better hover.
- Above ~2k nodes, switch to WebGL (`Scattergl`) and accept coarser visuals.
- Above ~10k nodes, leave Plotly behind — consider `datashader` rasterisation and keep interactivity for overviews only.
- Preserve the camera with `layout.uirevision` across every update.

### Exercises
1. Re-run the 2k example with plain `go.Scatter` and compare pan/zoom smoothness.
2. Increase to 10 000 nodes. At what size does hover become unusable?
3. Run an accessibility check: load the GOT composite with the Deuteranopia simulator (browser extension) and confirm the Okabe-Ito palette still works.

## 11. Further resources

**Plotly**
- Official Python reference — <https://plotly.com/python/reference/>
- Network graphs gallery — <https://plotly.com/python/network-graphs/>
- `FigureWidget` callbacks — <https://plotly.com/python/figurewidget/>
- Restyle vs. update vs. relayout — <https://plotly.com/python/update-buttons/>

**Pyvis**
- Docs — <https://pyvis.readthedocs.io/>
- `vis.js` physics options (the underlying engine) — <https://visjs.github.io/vis-network/docs/network/physics.html>

**Dash Cytoscape**
- Tutorial — <https://dash.plotly.com/cytoscape>
- Cytoscape.js layouts — <https://js.cytoscape.org/#layouts>

**Design & accessibility**
- Okabe-Ito palette rationale — <https://jfly.uni-koeln.de/color/>
- Kerren et al., *Information Visualization: Human-Centered Issues in Visual Representation* — chapter on interactive graph exploration.

## Final takeaway

Five questions a student should be able to answer after this notebook:

1. Why does Plotly build a network from two traces rather than a single `nx.draw()`-style call?
2. When does hover replace a static label, and when does a static annotation still win?
3. What's the difference between `updatemenus` and `FigureWidget`, and when do you reach for each?
4. Which interactive affordance collapses notebook 03's "which 8 labels?" problem entirely, and why?
5. At what graph size should you stop using Plotly, and what comes next?

The rule this notebook comes back to:

**Static tells a story. Interactive invites a question. Pick the one that matches what your audience actually needs to do.**